# Авторское решение: итоговое соревнование по модулю

Блокнот сравнивает несколько ансамблей, включая XGBoost, LightGBM и CatBoost, а затем строит финальный файл с вероятностями.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

target = "churn"
id_col = "id"


In [ ]:
def add_features(df):
    df = df.copy()
    df["sessions_per_100_days"] = df["sessions_last_30"] / (df["account_age_days"] / 100 + 1)
    df["price_after_discount"] = df["plan_price"] * (1 - df["discount_percent"] / 100)
    df["is_inactive"] = (df["days_since_last_activity"] >= 14).astype(int)
    df["support_per_session"] = df["support_tickets"] / (df["sessions_last_30"] + 1)
    df["failed_payment_flag"] = (df["failed_payments"] > 0).astype(int)
    df["high_price_no_discount"] = ((df["plan_price"] >= 1590) & (df["discount_percent"] <= 5)).astype(int)
    df["low_homework_many_tickets"] = ((df["homework_completion"] < 0.45) & (df["support_tickets"] >= 3)).astype(int)
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess(X_part):
    numeric_features = X_part.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_part.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_ohe()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


def make_pipeline(estimator, X_part):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocess(X_part)),
            ("model", estimator),
        ]
    )


def evaluate(name, estimator):
    model = make_pipeline(estimator, X_train)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "name": name,
        "ROC_AUC": roc_auc_score(y_val, proba),
        "average_precision": average_precision_score(y_val, proba),
        "F1_at_0_5": f1_score(y_val, pred, zero_division=0),
        "model": model,
        "val_proba": proba,
    }


In [ ]:
estimators = {
    "RandomForest": RandomForestClassifier(
        n_estimators=350,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=350,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=220,
        learning_rate=0.04,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=220,
        learning_rate=0.04,
        max_leaf_nodes=31,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=260,
        learning_rate=0.04,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=260,
        learning_rate=0.04,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=260,
        learning_rate=0.04,
        depth=4,
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
    ),
}

rows = []
trained = {}
val_predictions = {}

for name, estimator in estimators.items():
    result = evaluate(name, estimator)
    trained[name] = result.pop("model")
    val_predictions[name] = result.pop("val_proba")
    rows.append(result)

leaderboard = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
leaderboard


## Финальная смесь моделей

Для финального файла берём не просто первые строки одной validation-таблицы, а смесь разных семейств моделей: случайный лес, extremely randomized trees и CatBoost. Такая смесь показывает идею ансамблирования решений: модели ошибаются по-разному, и усреднение вероятностей иногда даёт более устойчивый результат.


In [ ]:
final_names = ["ExtraTrees", "CatBoost", "RandomForest"]
avg_val_proba = np.mean([val_predictions[name] for name in final_names], axis=0)

print("Усредняем модели:", final_names)
print("ROC-AUC среднего:", round(roc_auc_score(y_val, avg_val_proba), 4))
print("Average precision среднего:", round(average_precision_score(y_val, avg_val_proba), 4))


In [ ]:
final_probas = []
for name in final_names:
    estimator = clone(estimators[name])
    model = make_pipeline(estimator, X)
    model.fit(X, y)
    final_probas.append(model.predict_proba(test_fe[features])[:, 1])

test_proba = np.mean(final_probas, axis=0)

submission = pd.DataFrame(
    {
        "id": test_fe[id_col],
        "churn_probability": test_proba.round(5),
    }
)

submission.to_csv("author_submission.csv", index=False)
submission.head()
